In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [3]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/pre_cell_19.pickle

trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['cols_to_merge']
me:  13
trying: ['sample_discourse_id']
me:  2
trying: ['train_first_last']
me:  11
trying: ['av_per_essay']
me:  8
trying: ['train']
me:  14
trying: ['sample_submission']
me:  2
trying: ['ax2']
me:  8
trying: ['CountVectorizer']
me:  0
trying: ['np']
me:  0
trying: ['myid']
me:  12
trying: ['train_last']
me:  11
trying: ['stopwords']
me:  1
trying: ['cols_to_display']
me:  14
trying: ['myword']
me:  12
trying: ['ax1']
me:  8
trying: ['style']
me:  0
trying: ['last_ones']
me:  14
trying: ['IREWR_plug_2']
me:  16
trying: ['word_dict']
me:  12
trying: ['ax']
me:  11
trying: ['FuncFormatter']
me:  0
trying: ['IREWR_tmp2']
me:  17
trying: ['total_gaps']
me:  19
trying: ['all_gaps']
me:  18
trying: ['data']
me:  12
trying: ['txt_file']
me:  12
trying: ['t']
me:  12
trying: ['IREWR_tmp']
me:  16
trying: ['glob']
me:  0
trying: ['pd']
me:  0
trying: ['train_first']
me:  11
trying: ['tqdm']


In [4]:
%%RecordEvent
%%time
### cell 19 ###
def add_gap_rows(essay):
    cols = ['discourse_start', 'discourse_end', 'discourse_type', 'gap_length', 'gap_end_length']
    # pull out this essay's rows
    df_essay = train.query(f"id == '{essay}'")[cols].reset_index(drop=True)
    print(df_essay)

    # build any interior gaps
    new_rows = []
    # for each row from the second onward, if there's a gap before it, record that gap
    for i in range(1, len(df_essay)):
        if df_essay.loc[i, 'gap_length'] > 0:
            start = df_essay.loc[i-1, 'discourse_end'] + 1
            end   = df_essay.loc[i, 'discourse_start'] - 1
            new_rows.append([start, end, 'Nothing', np.nan, np.nan])

    # build a final gap if the last row has one
    last = len(df_essay) - 1
    if last >= 0 and df_essay.loc[last, 'gap_end_length'] > 0:
        start = df_essay.loc[last, 'discourse_end'] + 1
        end   = start + df_essay.loc[last, 'gap_end_length']
        new_rows.append([start, end, 'Nothing', np.nan, np.nan])

    # if we found any gap rows, concat them in
    if new_rows:
        gap_df = pd.DataFrame(new_rows, columns=cols)
        df_essay = pd.concat([df_essay, gap_df], ignore_index=True)

    # sort by start and reset the index
    df_essay = df_essay.sort_values('discourse_start').reset_index(drop=True)
    return df_essay

# example call (will now match the original exactly)
add_gap_rows(IREWR_plug_1)

   discourse_start  discourse_end discourse_type  gap_length  gap_end_length
0                0            778           Lead         NaN             NaN
1              779            999       Position         NaN             NaN
2             1000           1117          Claim         NaN             NaN
3             1118           1350       Evidence         NaN             NaN
4             1351           1466          Claim         NaN             NaN
5             1467           1672       Evidence         NaN           862.0
CPU times: user 408 ms, sys: 35.4 ms, total: 444 ms
Wall time: 447 ms


,discourse_start,discourse_end,discourse_type,gap_length,gap_end_length
0,0,778.0,Lead,NaN,NaN
1,779,999.0,Position,NaN,NaN
2,1000,1117.0,Claim,NaN,NaN
3,1118,1350.0,Evidence,NaN,NaN
4,1351,1466.0,Claim,NaN,NaN
5,1467,1672.0,Evidence,NaN,862.0
6,1673,2535.0,Nothing,<NA>,<NA>


In [5]:
%Checkpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_19_try_7.pickle

migration speed (bps): 754321732.667513
---------------------------
variables to migrate:
i_1 28
CountVectorizer 1072
counter 28
IREWR_plug_2 61
FuncFormatter 1072
ax 48
ax2 48
warnings 72
sample_submission 48
ax1 48
np 72
sample_discourse_id 32
all_gaps 48
cols_to_display 120
av_per_essay 48
IREWR_tmp2 48
train_first_last 48
train 48
last_ones 48
IREWR_plug_1 61
i_3 28
cols_to_merge 120
glob 144
pd 72
test_txt 120
myid 61
stopwords 48
data 2813
train_txt 126104
txt_file 208
t 166
IREWR_tmp 48
myword 28
tqdm 1072
len_dict 589920
train_first 48
word_dict 589920
train_last 48
mylen 28
total_gaps 48
plt 72
add_gap_rows 144
style 72
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/cell_19/checkpoints/post_cell_19_try_7.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables []
Active variables ['sample_submission', 'train_txt', 'train']
Intermediate variables ['stopwords', 'test_txt']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables []
Active variables ['sample_discourse_id', 'train']
Intermediate variables ['sample_submission']
Future variables ['train_txt']
Modified dataframes
  train
    Input columns: set()
    Changed columns: {'discourse_id', 'discourse_text', 'discourse_type', 'discourse_start', 'predictionstring', 'id', 'discourse_end', 'discourse_type_num'}
    Created columns: set()
    Deleted columns: set()
  sample_submission
    Input columns: set()
    Changed columns: {'id', 'predictionstring', 'class'}
    Created columns: set()
    Deleted columns: set()
======= Cell 2 =======
Input variables []
Active variables ['cols_to_display', 'train']
Intermediate variables []
Future variables ['sample_discourse_id', 'train_txt']
Modified dataframes
  train
    Input columns: set(

In [7]:

with open("/home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_19_try_7.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[19], f)


In [8]:
opt_output = Out.get(4)

In [9]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated/checkpoints/post_cell_19.pickle

import numpy as np
import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO(jie): this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['i_3', 'i_1']
me:  13
me:  11
trying: ['train_first_last']
me:  11
trying: ['plt']
me:  0
trying: ['orig_output']
me:  21
trying: ['style']
me:  0
trying: ['CountVectorizer']
me:  0
trying: ['counter']
me:  11
trying: ['IREWR_plug_2']
me:  16
trying: ['FuncFormatter']
me:  0
trying: ['ax2']
me:  8
trying: ['ax']
me:  11
trying: ['warnings']
me:  0
trying: ['sample_submission']
me:  2
trying: ['ax1']
me:  8
trying: ['np']
me:  0
trying: ['sample_discourse_id']
me:  2
trying: ['cols_to_display']
me:  14
trying: ['all_gaps']
me:  20
trying: ['last_ones']
me:  20
trying: ['train']
me:  14
trying: ['av_per_essay']
me:  20
trying: ['total_gaps']
me:  20
trying: ['IREWR_plug_1']
me:  17
trying: ['IREWR_tmp2']
me:  20
trying: ['IREWR_tmp']
me:  20
trying: ['cols_to_merge']
me:  13
trying: ['glob']
me:  0
trying: ['add_gap_rows']
me:  20
trying: ['pd']
me:  0
trying: ['test_txt']
me:  1
trying: ['myid']
me:  12
trying: ['stopwords']
me:  1
trying:

AssertionError: 